# Microsoft Agent Framework — Azure OpenAI (API delle Risposte)

In questo esempio di codice, userai il **Microsoft Agent Framework (MAF)** per creare un agente semplice supportato da **Azure OpenAI** utilizzando l'**API delle Risposte**.

> **Nota sulla migrazione:** Questo esempio utilizzava in precedenza Semantic Kernel con GitHub Models. È stato migrato al Microsoft Agent Framework, e GitHub Models (deprecato, in pensionamento a luglio 2026) è stato sostituito con Azure OpenAI, che supporta l'API delle Risposte. `OpenAIChatClient` in MAF punta all'endpoint stabile `/openai/v1/` di Azure OpenAI e usa l'API delle Risposte per impostazione predefinita.

Lo scopo di questo esempio è dimostrare i passaggi che saranno applicati in seguito in ulteriori esempi di codice durante l'implementazione di vari schemi agentici.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Importa i pacchetti Python necessari


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Definire uno Strumento

Nel Microsoft Agent Framework, uno **strumento** è una semplice funzione Python decorata con `@tool` che l'agente può chiamare. Di seguito definiamo uno strumento che restituisce una destinazione vacanziera casuale ed evita di ripetere quella precedente.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Creazione dell'Agente

Qui, creiamo l'Agente chiamato `TravelAgent`.

In questo esempio, usiamo istruzioni molto semplici. Sentitevi liberi di modificare queste istruzioni per osservare come cambia il comportamento dell'agente.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Esecuzione dell'agente

Ora possiamo eseguire l'agente. Creiamo una `AgentSession` in modo che l'agente ricordi la conversazione tra i turni, quindi inviamo due `user_inputs`. Il primo chiede un viaggio; il secondo dice che all'utente non è piaciuto il suggerimento e chiede un altro — l'agente usa la cronologia della sessione più lo strumento `get_random_destination` per rispondere.

Puoi modificare questi messaggi per osservare come l'agente reagisce in modo diverso. Le risposte sono **streamed** token per token.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
